# 04 · Modelos PyTorch

Dois modelos, ambos prevendo as **variações** da cota nos 3 horizontes de uma vez (saída com 3 valores):

- **MLP** sobre as features tabulares;
- **GRU** sobre as últimas 48 h de todas as variáveis.

Entradas e alvos são padronizados com média/desvio **do treino**. A validação serve só para o *early stopping*; o teste (eventos extremos) não é usado no treino.
Para execução rápida de teste: `CAPSTONE_FAST=1` (poucas épocas).

In [ ]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # esconde a GPU; precisa vir antes de importar o torch
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, data, features, metrics, pipeline
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True

import torch, json
from src import models

FAST = os.environ.get("CAPSTONE_FAST") == "1"
EPOCHS = 3 if FAST else 80
N_SEEDS = 1 if FAST else 3          # média de várias sementes deixa o resultado mais estável
DEVICE = torch.device("cpu")
print("dispositivo:", DEVICE, "| épocas:", EPOCHS, "| sementes:", N_SEEDS)

In [ ]:
def treinar_conjunto(make_model, Xtr, Ytr, Xva, Yva, Xall, nome, alvo, epochs=EPOCHS, n_seeds=N_SEEDS, **fit_kw):
    """Treina n_seeds modelos e devolve a média das previsões (em escala padronizada) + histórico da 1ª semente."""
    preds, hist0 = [], None
    for s in range(n_seeds):
        models.set_seed(config.RANDOM_STATE + s)
        m = make_model()
        m, hist = models.fit(m, Xtr, Ytr, Xva, Yva, epochs=epochs, device=DEVICE, verbose=(s == 0), **fit_kw)
        torch.save(m.state_dict(), config.MODELS / f"{nome}_{alvo}_seed{s}.pt")
        preds.append(models.predict(m, Xall, device=DEVICE))
        hist0 = hist0 or hist
    return np.mean(preds, axis=0), hist0

In [ ]:
panel = pipeline.load_panel()
resultados = {}
historicos = {}

for alvo in config.TARGETS:
    if f"cota_{alvo}" not in panel:
        continue
    print(f"\n################ {config.STATIONS[alvo]['nome']} ({alvo}) ################")

    # ---------- MLP (tabular)
    P = features.build_tabular(panel, alvo)
    tr, va, te = (P.masks[k] for k in ("train", "val", "test"))
    Xs, _ = features.nn_inputs(P)
    sy = features.Scaler.fit(P.Y[tr])
    Ys = sy.transform(P.Y)
    pred_s, historicos[("mlp", alvo)] = treinar_conjunto(
        lambda: models.MLP(Xs.shape[1], Ys.shape[1], hidden=(128, 64), dropout=0.1),
        Xs[tr], Ys[tr], Xs[va], Ys[va], Xs, "mlp", alvo)
    pred_abs_mlp = P.base[:, None] + sy.inverse(pred_s)
    pipeline.save_predictions("mlp", alvo, P, pred_abs_mlp)

    # ---------- GRU (sequências)
    S = features.build_sequences(panel, alvo)
    tr2, va2 = S.masks["train"], S.masks["val"]
    Xq, _ = features.nn_inputs(S)
    sys_ = features.Scaler.fit(S.Y[tr2])
    Yq = sys_.transform(S.Y)
    pred_s2, historicos[("gru", alvo)] = treinar_conjunto(
        lambda: models.GRUNet(Xq.shape[2], Yq.shape[1], hidden=64, num_layers=1, dropout=0.1),
        Xq[tr2], Yq[tr2], Xq[va2], Yq[va2], Xq, "gru", alvo)
    pred_abs_gru = S.base[:, None] + sys_.inverse(pred_s2)
    pipeline.save_predictions("gru", alvo, S, pred_abs_gru)

    # ---------- métricas no teste
    tabs = []
    for nome, PP, pa in (("mlp", P, pred_abs_mlp), ("gru", S, pred_abs_gru)):
        m = PP.masks["test"]
        if m.sum() == 0:
            continue
        t = metrics.summary_table(PP.Yabs[m], pa[m], PP.base[m], PP.horizons)
        t.insert(0, "modelo", nome); tabs.append(t.reset_index())
    if tabs:
        resultados[alvo] = pd.concat(tabs).set_index(["modelo", "horizonte_h"])
        display(resultados[alvo])

## Curvas de treino (perda padronizada)

In [ ]:
fig, axes = plt.subplots(1, len(historicos), figsize=(4.2 * len(historicos), 3.2), squeeze=False)
for ax, ((nome, alvo), h) in zip(axes[0], historicos.items()):
    ax.plot(h["train"], label="treino"); ax.plot(h["val"], label="validação")
    ax.set_title(f"{nome} · {alvo}"); ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.show()

**Atenção ao interpretar:** o MLP e a GRU usam `skill_vs_persist` como o baseline: se ficarem abaixo do gradient boosting do notebook 03, isso também é um resultado válido.
Redes recorrentes costumam render mais com dados de **muitas bacias**; com uma bacia só, árvores e modelos simples frequentemente competem bem.
Os pesos ficam em `outputs/models/`.